# TEKNOFEST — Sağlıkta Yapay Zeka
## Per-panel variant pathogenicity pipeline (Google Colab)

This notebook runs the full per-panel pipeline end to end and shows the per-panel
**binary-F1** scores:

> feature selection (null-importance + top-K floor) → Optuna tuning → calibrated
> stacking → per-panel champion selection → inference wiring check

### How to run — step by step
1. **Enable the GPU runtime:** menu **Runtime ▸ Change runtime type ▸ Hardware accelerator = GPU (T4) ▸ Save**.
2. **Put the project on Google Drive:** upload the whole `TEKNOFEST-SagliktaYapayZeka`
   folder (it already contains the dataset under `universite-veri-seti/`) to **My Drive**.
3. **Run every cell top to bottom.** If Colab shows a **"Restart runtime"** button after the
   install cell, click it, then continue from the **Mount Drive** cell (do *not* re-run install).
4. Scores appear in **Section 8 — Results**.

### A note on the GPU (please read)
The four panels are tiny tabular datasets (111–2931 rows, ~30–50 features after selection).
XGBoost / LightGBM / CatBoost each fit in well under a second on CPU, and the two diverse
bases (ExtraTrees, L1-logistic) are scikit-learn — **CPU-only**. A GPU gives **no meaningful
speedup** for this workload, so the pipeline runs on CPU for exact reproducibility. The GPU is
still enabled (Step 1) and confirmed below; Section 11 shows how to switch the boosters to GPU
if you want to experiment.


## 1. Requirements
Pinned to the project's frozen environment so scikit-learn 1.9 (TargetEncoder, the
`l1_ratio` API), pandas 3.0 and the boosters behave exactly as in development.

In [ ]:
# Requirements (pinned). One line so there are no shell-continuation surprises.
%pip install -q numpy==2.4.6 pandas==3.0.3 scipy==1.17.1 scikit-learn==1.9.0 xgboost==3.2.0 lightgbm==4.6.0 catboost==1.2.10 optuna==4.9.0 joblib==1.5.3 biopython==1.87
print("[done] If a 'RESTART RUNTIME' button appeared above, click it, then resume from the "
      "'Mount Drive' cell below (do NOT re-run this install cell).")

## 2. Confirm the GPU is available

In [ ]:
!nvidia-smi -L || echo "No GPU detected -> Runtime > Change runtime type > GPU."
import sys
print("Python:", sys.version.split()[0])

## 3. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 4. Locate the project and copy it to fast local disk
Set **`DRIVE_PROJECT`** to the folder you uploaded in Step 2. We copy it to `/content`
because Google Drive's filesystem is slow for the many small files the pipeline writes.

If your dataset lives **somewhere else** on Drive (not inside the project folder), set
`TEKNOFEST_DATA_DIR` to the folder that contains the four `YARISMA_TRAIN_*.csv` files.

In [ ]:
import os, shutil, pathlib

# >>> EDIT to match where you uploaded the folder on your Drive <<<
DRIVE_PROJECT = "/content/drive/MyDrive/TEKNOFEST-SagliktaYapayZeka"

LOCAL_PROJECT = "/content/TEKNOFEST-SagliktaYapayZeka"
assert os.path.isdir(DRIVE_PROJECT), f"Not found on Drive: {DRIVE_PROJECT!r} — fix DRIVE_PROJECT."

if os.path.exists(LOCAL_PROJECT):
    shutil.rmtree(LOCAL_PROJECT)
shutil.copytree(DRIVE_PROJECT, LOCAL_PROJECT,
                ignore=shutil.ignore_patterns('.git', '.venv', '__pycache__', 'outputs'))
os.chdir(LOCAL_PROJECT)
print("Working dir:", os.getcwd())

# Dataset: use the copy inside the project, or override if it lives elsewhere on Drive.
default_data = pathlib.Path(LOCAL_PROJECT) / "universite-veri-seti" / "EĞİTİM (TRAIN) SETLERİ"
if not default_data.is_dir():
    os.environ["TEKNOFEST_DATA_DIR"] = "/content/drive/MyDrive/TEKNOFEST-data/EĞİTİM (TRAIN) SETLERİ"
print("Data dir:", os.environ.get("TEKNOFEST_DATA_DIR", str(default_data)))

## 5. Sanity check — load each panel from Drive

In [ ]:
import sys
sys.path.insert(0, os.getcwd())
from src import config, data_io
print("Data dir resolved to:", config.DATA_DIR)
print()
for p in config.PANELS:
    df = data_io.load_panel(p)
    pos = (df['Label'] == 1).mean()
    print(f"  {p:7s} rows={len(df):5d}  cols={df.shape[1]}  pathogenic={pos:.1%}")

## 6. Run settings
`QUICK = True` runs a fast preview (~25–40 min). Set `QUICK = False` to reproduce the
committed champion F1 (full Optuna budget; ~60–90 min).

In [ ]:
QUICK = True   # set False for the full, report-grade run

import os
os.environ["OPTUNA_TRIALS"]  = "12" if QUICK else "40"
os.environ["OPTUNA_TIMEOUT"] = "90" if QUICK else "600"
os.environ["PYTHONPATH"] = os.getcwd()
print("QUICK =", QUICK,
      "| OPTUNA_TRIALS =", os.environ["OPTUNA_TRIALS"],
      "| OPTUNA_TIMEOUT =", os.environ["OPTUNA_TIMEOUT"], "s/study")

## 7. Run the pipeline
Each step writes to `outputs/`. The run is deterministic (fixed seed). The Optuna step is the
long pole. You can watch progress in each cell's output.

In [ ]:
# 7a. Feature selection (null-importance + panel-size-bounded top-K floor)
!PYTHONPATH=. python train/run_feature_selection.py

In [ ]:
# 7b. Optuna tuning (5 base models x 4 panels). This is the long step.
!PYTHONPATH=. python train/run_optuna.py

In [ ]:
# 7c. Calibrated stacking + per-panel champion selection
!PYTHONPATH=. python train/run_stacking.py
!PYTHONPATH=. python train/finalize_champions.py

## 8. Results — per-panel champion F1
`oof_f1` is the honest out-of-fold binary-F1 on the Pathogenic class. `ref_f1` is the
committed reference; `delta_vs_ref` shows movement (expect ~0 to slightly negative on
`QUICK`, and ≥ the reference on the full run).

In [ ]:
import pandas as pd
champs = pd.read_csv("outputs/models/champions_summary.csv")
stack  = pd.read_csv("outputs/stacking/stacking_summary.csv")

ref = {"GENEL": 0.9032, "KANSER": 0.9446, "PAH": 0.9464, "CFTR": 0.9783}
champs["ref_f1"] = champs["panel"].map(ref)
champs["delta_vs_ref"] = (champs["oof_f1"] - champs["ref_f1"]).round(4)
view = champs[["panel", "strategy", "model", "threshold", "oof_f1", "ref_f1", "delta_vs_ref"]].copy()
view["oof_f1"] = view["oof_f1"].round(4)
view["threshold"] = view["threshold"].round(3)
print("=== Per-panel champion (honest OOF binary-F1) ===")
display(view)
print(f"\nMean champion F1: {champs['oof_f1'].mean():.4f}")
print("\n=== Stacking detail (single vs mean vs stacked variants) ===")
display(stack.round(4))

## 9. Inference wiring check (`predict.py`)
Runs the saved champion bundle per panel on the training CSV. The printed F1 is **in-sample**
(a wiring check, not an honest score); the predicted positive rate should track the base rate.

In [ ]:
for p in config.PANELS:
    print("-" * 64)
    !PYTHONPATH=. python predict.py {p}

## 10. (Optional) Save the trained bundles + scores back to Drive

In [ ]:
import shutil, os
dst = os.path.join(DRIVE_PROJECT, "outputs")
if os.path.isdir("outputs"):
    shutil.copytree("outputs", dst, dirs_exist_ok=True)
    print("Saved outputs to:", dst)
else:
    print("No outputs/ to save yet — run Section 7 first.")

## 11. (Optional) Run the gradient boosters on GPU
Not recommended for this dataset — CPU is faster here and GPU results differ slightly — but if
you want to use the GPU for the boosters, these are the one-line switches in the model factories:

- **XGBoost** (`src/models.py`, `src/tuning.py`): add `device="cuda"` (keep `tree_method="hist"`).
- **CatBoost** (`src/models.py`, `src/tuning.py`): add `task_type="GPU"`.
- **LightGBM**: add `device="gpu"` — requires a GPU-enabled LightGBM build.

ExtraTrees and L1-logistic are scikit-learn and stay on CPU.

### Troubleshooting
- **"Restart runtime" after install** — click it, then resume from *Section 3 (Mount Drive)*.
- **`Not found on Drive`** — fix `DRIVE_PROJECT` (left sidebar ▸ Files ▸ drive ▸ MyDrive).
- **Dataset not found** — set `TEKNOFEST_DATA_DIR` (Section 4) to the folder with the four
  `YARISMA_TRAIN_*.csv` files.
- **Prefer GitHub over Drive for the code?** Replace Section 4's copy with
  `!git clone <your-repo-url> /content/TEKNOFEST-SagliktaYapayZeka` and keep the dataset on Drive
  via `TEKNOFEST_DATA_DIR`.
- **Too slow / out of RAM** — keep `QUICK = True`.
